# WD Tagger API Server on Colab

通过 ngrok 暴露 onnxruntime GPU 推理 API。
选一个模型，运行后你会得到一个外网可访问的 API 地址。

In [1]:
# @title 1. 安装依赖
!pip install -q onnxruntime-gpu fastapi uvicorn pyngrok pillow numpy nest-asyncio
# ⚠️ 填入你自己的 ngrok authtoken:
# !ngrok authtoken YOUR_TOKEN_HERE


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 1.6 MB/s eta 0:00:00
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [2]:
# @title 2. 下载模型
import os, urllib.request

# @markdown 可选模型:
# @markdown - `SmilingWolf/wd-eva02-large-tagger-v3` (最大最准, ~1.2GB)
# @markdown - `SmilingWolf/wd-vit-tagger-v3` (轻量快速, ~350MB)
# @markdown - `SmilingWolf/wd-swinv2-tagger-v3` (均衡, ~500MB)
MODEL_ID = "SmilingWolf/wd-eva02-large-tagger-v3"  # @param ["SmilingWolf/wd-eva02-large-tagger-v3", "SmilingWolf/wd-vit-tagger-v3", "SmilingWolf/wd-swinv2-tagger-v3"]

BASE = f"https://huggingface.co/{MODEL_ID}/resolve/main"
os.makedirs("/content/model", exist_ok=True)

for f in ["model.onnx", "selected_tags.csv", "config.json"]:
    path = f"/content/model/{f}"
    if not os.path.exists(path):
        print(f"Downloading {f}...")
        urllib.request.urlretrieve(f"{BASE}/{f}", path)
        print(f"  OK ({os.path.getsize(path)/1024/1024:.0f} MB)")
    else:
        print(f"{f} 已存在")

print("\n模型就绪")

  OK (1202 MB)
  OK (0 MB)
  OK (0 MB)

模型就绪


In [3]:

# @title 3. 启动 API Server
import io, csv, time, numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import onnxruntime as ort

# 自动使用 Cell 2 下载的模型
model_path = "/content/model/model.onnx"
csv_path = "/content/model/selected_tags.csv"

session = ort.InferenceSession(model_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
_, target_size, _, _ = session.get_inputs()[0].shape

tags_list, cat_idx = [], {"rating": [], "general": [], "character": []}
with open(csv_path) as f:
    for i, row in enumerate(csv.DictReader(f)):
        cat = int(row["category"])
        tags_list.append({"name": row["name"], "category": cat})
        if cat == 9: cat_idx["rating"].append(i)
        elif cat == 0: cat_idx["general"].append(i)
        elif cat == 4: cat_idx["character"].append(i)

def preprocess(img):
    if img.mode == "RGBA":
        canvas = Image.new("RGBA", img.size, (255, 255, 255))
        canvas.alpha_composite(img)
        img = canvas.convert("RGB")
    else:
        img = img.convert("RGB")
    w, h = img.size; m = max(w, h)
    padded = Image.new("RGB", (m, m), (255, 255, 255))
    padded.paste(img, ((m-w)//2, (m-h)//2))
    if m != target_size:
        padded = padded.resize((target_size, target_size), Image.BICUBIC)
    arr = np.asarray(padded, dtype=np.float32)[:, :, ::-1]
    return np.expand_dims(arr, 0)

def infer(img, gt=0.35, ct=0.85):
    t0 = time.time()
    probs = session.run([output_name], {input_name: preprocess(img)})[0][0]
    t = time.time() - t0
    rating = {tags_list[i]["name"]: float(probs[i]) for i in cat_idx["rating"]}
    tags = [tags_list[i]["name"] for i in cat_idx["general"] if probs[i] >= gt]
    chars = [tags_list[i]["name"] for i in cat_idx["character"] if probs[i] >= ct]
    return {"rating": max(rating, key=rating.get), "rating_detail": rating, "tags": tags, "characters": chars, "time": round(t, 2)}

app = FastAPI()

@app.get("/")
def root():
    return {"status": "ok"}

@app.post("/tag")
async def tag(file: UploadFile = File(...), gt: float = 0.35, ct: float = 0.85):
    try:
        return infer(Image.open(io.BytesIO(await file.read())), gt, ct)
    except Exception as e:
        import traceback
        return JSONResponse({"error": str(e), "trace": traceback.format_exc()}, 500)

import asyncio, nest_asyncio, uvicorn; nest_asyncio.apply()
from pyngrok import ngrok
try:
    ngrok.kill()
    url = ngrok.connect(8000, bind_tls=True).public_url
except:
    url = "???"
print(f"\n🚀 {url}")
cfg = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
asyncio.get_event_loop().create_task(uvicorn.Server(cfg).serve())
print("✅ 就绪")


🚀 https://aftermost-equator-labrador.ngrok-free.dev
✅ 就绪
